<a href="https://colab.research.google.com/github/alj-x64/Realtime-Bass-Tablature-Transcription/blob/main/Colab_Master_Training_Script.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
import time
import numpy as np
import optuna
import csv
import os

# Google Colab specific import para sa Drive mounting
try:
    from google.colab import drive
except ImportError:
    print("[WARNING] Hindi ka nasa Google Colab environment. Magse-save locally.")

from adaptive_hpo import AdaptiveMultiStageStressTestedHPO
from tabcnn_model import DynamicTabCNN
from dataset_loader import ISBDataset

# ==========================================
# 1. THE SHARED PYTORCH TRAINING PIPELINE
# ==========================================
def evaluate_model_pipeline(config, train_loader=None, val_loader=None, stress_test=False, profile_latency=True):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = DynamicTabCNN(config).to(device)
    
    criterion_cls = nn.CrossEntropyLoss()
    criterion_bin = nn.BCELoss()
    
    optimizer = optim.Adam(model.parameters(), lr=config['learning_rate'])
    
    max_epochs = 50 
    patience = 5    
    best_val_loss = float('inf')
    patience_counter = 0
    
    if train_loader is not None and val_loader is not None:
        for epoch in range(max_epochs):
            model.train()
            for batch_audio, labels in train_loader:
                batch_audio = batch_audio.to(device)
                labels = {k: v.to(device) for k, v in labels.items()}
                
                out_s, out_f, out_p, out_on, out_off = model(batch_audio)
                
                loss_s = criterion_cls(out_s, labels['string'])
                loss_f = criterion_cls(out_f, labels['fret'])
                loss_p = criterion_cls(out_p, labels['pitch'])
                
                loss_on = criterion_bin(out_on, labels['onset'])
                loss_off = criterion_bin(out_off, labels['offset'])
                
                total_loss = loss_s + loss_f + loss_p + loss_on + loss_off
                
                optimizer.zero_grad()
                total_loss.backward()
                optimizer.step()
                
            model.eval()
            val_loss_accum = 0.0
            with torch.no_grad():
                for val_audio, val_labels in val_loader:
                    val_audio = val_audio.to(device)
                    val_labels = {k: v.to(device) for k, v in val_labels.items()}
                    
                    v_out_s, v_out_f, v_out_p, v_out_on, v_out_off = model(val_audio)
                    
                    v_loss_s = criterion_cls(v_out_s, val_labels['string'])
                    v_loss_f = criterion_cls(v_out_f, val_labels['fret'])
                    v_loss_p = criterion_cls(v_out_p, val_labels['pitch'])
                    v_loss_on = criterion_bin(v_out_on, val_labels['onset'])
                    v_loss_off = criterion_bin(v_out_off, val_labels['offset'])
                    
                    val_loss_accum += (v_loss_s + v_loss_f + v_loss_p + v_loss_on + v_loss_off).item()
                    
            epoch_val_loss = val_loss_accum / len(val_loader)
            
            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                patience_counter = 0 
            else:
                patience_counter += 1 
                
            if patience_counter >= patience:
                break

    validation_loss = best_val_loss if best_val_loss != float('inf') else 1.0
    avg_latency = 0.0 
    
    if profile_latency:
        model.eval()
        dummy_input = torch.randn(1, 4608).to(device)
            
        latencies = []
        with torch.no_grad():
            for _ in range(10): 
                if stress_test:
                    noise_std = 0.05 
                    network_input = dummy_input + torch.randn_like(dummy_input) * noise_std
                else:
                    network_input = dummy_input

                t_i = time.perf_counter()
                _ = model(network_input) 
                t_o = time.perf_counter()
                
                latencies.append((t_o - t_i) * 1000)
                
        avg_latency = sum(latencies) / len(latencies)
            
    return validation_loss, avg_latency

def create_optuna_objective(train_loader, val_loader, profile_latency=False):
    def optuna_objective(trial):
        config = {
            'learning_rate': trial.suggest_float('learning_rate', 1e-5, 1e-2, log=True),
            'dropout_rate': trial.suggest_float('dropout_rate', 0.1, 0.5),
            'activation': trial.suggest_categorical('activation', ['ReLU', 'Tanh', 'ELU']),
            'conv_layers': trial.suggest_int('conv_layers', 1, 4),
            'filter_layers': trial.suggest_categorical('filter_layers', [16, 32, 64, 128]),
            'kernel_size': trial.suggest_categorical('kernel_size', [2, 3, 5, 7])
        }
        loss, latency = evaluate_model_pipeline(config, train_loader, val_loader, stress_test=False, profile_latency=profile_latency)
        return 1.0 - loss 
    return optuna_objective

def optuna_csv_logger(study, trial, filename):
    file_exists = os.path.isfile(filename)
    with open(filename, mode='a', newline='') as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(['Trial', 'LearningRate', 'DropoutRate', 'Activation', 
                             'ConvLayers', 'FilterLayers', 'KernelSize', 'Fitness'])
        writer.writerow([trial.number + 1, f"{trial.params['learning_rate']:.6f}",
                         trial.params['dropout_rate'], trial.params['activation'],
                         trial.params['conv_layers'], trial.params['filter_layers'],
                         trial.params['kernel_size'], f"{trial.value:.4f}"])

# ==========================================
# 3. MAIN EXECUTION BLOCK
# ==========================================
if __name__ == "__main__":
    print("--- Thesis Optimization Experiments ---")
    
    # 0. MOUNT GOOGLE DRIVE & SETUP DIRECTORY
    try:
        drive.mount('/content/drive')
        GDRIVE_PATH = "/content/drive/MyDrive/CNN Training"
    except:
        GDRIVE_PATH = "./CNN Training" # Fallback kung offline testing
        
    if not os.path.exists(GDRIVE_PATH):
        os.makedirs(GDRIVE_PATH)
        print(f"[STORAGE SETUP] Gumawa ng bagong folder sa: {GDRIVE_PATH}")
    else:
        print(f"[STORAGE SETUP] Naka-link na sa existing folder: {GDRIVE_PATH}")

    MODE = "PROPOSED" 
    TOTAL_TRIALS = 30 
    
    print("\n[1] Naglo-load ng Dataset...")
    full_dataset = ISBDataset(csv_file="dataset_labels.csv", root_dir="./IDMT-SMT-BASS")
    
    train_size = int(0.8 * len(full_dataset))
    val_size = len(full_dataset) - train_size
    train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])
    
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
    
    print(f"\n[2] Sinisimulan ang Optimization Mode: {MODE}")
    if MODE == "PROPOSED":
        # Ikabit sa GDrive ang log file at checkpoint file
        hpo_log_path = os.path.join(GDRIVE_PATH, "hpo_proposed_log.csv")
        checkpoint_path = os.path.join(GDRIVE_PATH, "hpo_state.pkl")
        
        hpo_engine = AdaptiveMultiStageStressTestedHPO(
            budgetTrial=TOTAL_TRIALS, alpha=0.5, beta=0.5, rho_S=0.60, rho_R=0.40, gamma=0.30,
            logfile=hpo_log_path, 
            checkpoint_file=checkpoint_path # Naka-link na sa GDrive folder mo
        )
        train_eval_wrapper = lambda config, stress_test=False: evaluate_model_pipeline(
            config, train_loader, val_loader, stress_test=stress_test, profile_latency=True
        )
        best_config = hpo_engine.optimization(train_eval_wrapper)
    
    elif MODE in ["RANDOM", "BAYESIAN"]:
        # Ikabit sa GDrive ang SQLite Database at CSV logs
        db_path = f"sqlite:///{os.path.join(GDRIVE_PATH, f'optuna_{MODE.lower()}_study.db')}"
        log_file = os.path.join(GDRIVE_PATH, f"optuna_{MODE.lower()}_log.csv")
        
        sampler = optuna.samplers.RandomSampler() if MODE == "RANDOM" else optuna.samplers.TPESampler()
        
        study = optuna.create_study(
            study_name=f"thesis_{MODE.lower()}", 
            storage=db_path, 
            load_if_exists=True, 
            direction="maximize", 
            sampler=sampler
        )
        
        objective_function = create_optuna_objective(train_loader, val_loader)
        
        trials_left = TOTAL_TRIALS - len(study.trials)
        if trials_left > 0:
            print(f"Resuming Optuna... May {trials_left} trials pa na natitira.")
            study.optimize(objective_function, n_trials=trials_left, callbacks=[lambda s, t: optuna_csv_logger(s, t, log_file)])
        else:
            print("Optuna study is already completed.")
            
        best_config = study.best_params

    # ==========================================
    # 4. FINAL RETRAINING AND MODEL CHECKPOINTING
    # ==========================================
    print(f"\n>>> FINAL RETRAINING WITH BEST CONFIG: {best_config}")
    
    final_model = DynamicTabCNN(best_config).to('cuda' if torch.cuda.is_available() else 'cpu')
    
    # I-save ang Final Jetson Nano Weights direkta sa GDrive folder mo
    model_save_path = os.path.join(GDRIVE_PATH, f"trained_{MODE.lower()}.pth")
    torch.save(final_model.state_dict(), model_save_path)
    
    print(f"✅ Na-save na ang trained weights sa iyong Google Drive: {model_save_path}")